# MentalHealth R1 Existing-Data Analyses

Reviewer-response analyses that can be computed from existing prediction and annotation outputs: paired hard-vs-soft tests, aspect-level per-class metrics, entropy triage thresholds, and LLM robustness-design summaries from the separate GTheory subset.

In [1]:
from pathlib import Path
import os
import pandas as pd

REPO_ROOT = Path.cwd()
if REPO_ROOT.name.startswith('6. Revision'):
    REPO_ROOT = REPO_ROOT.parent

DATA_ROOT = Path(os.environ.get('DATA_ROOT', REPO_ROOT / '0. Dataset'))

# Expected after downloading the OSF review data package:
# 0. Dataset/existing_data_analyses/*.csv
# The fallback paths support either a compact OSF layout or an outputs-style layout.
existing_candidates = [
    DATA_ROOT / 'existing_data_analyses',
    DATA_ROOT / 'outputs' / 'existing_data_analyses',
]
OUT_DIR = next((p for p in existing_candidates if p.exists()), existing_candidates[0])
OUT_DIR


(PosixPath('./03. Revision_R1'), PosixPath('./03. Revision_R1/03_existing_data_analyses/outputs'))

In [2]:
# Re-run all existing-data analyses.
%run "$SCRIPT"

## Paired Hard-vs-Soft Tests

This addresses the reviewer concern that overlapping CIs are not a paired significance test. The hard-label and soft-label prediction files are aligned by statement and true AI hard label before computing paired bootstrap deltas and McNemar tests.

In [3]:
paired = pd.read_csv(OUT_DIR / 'paired_hard_vs_soft_tests.csv')
paired

,model,hard_weighted_f1,soft_weighted_f1,paired_delta_f1,bootstrap_delta_mean,bootstrap_delta_ci_low,bootstrap_delta_ci_high,mcnemar_p,mcnemar_table
0,ALBERT,0.784285,0.793661,0.009376,0.009411,0.003181,0.015815,3.441839e-08,"[[7694, 509], [702, 1645]]"
1,BioBERT,0.776939,0.784704,0.007766,0.007676,0.001275,0.014228,5.135671e-06,"[[7627, 521], [680, 1722]]"


## Aspect-Level Imbalance-Aware Metrics

This reports NONE/WEAK/CLEAR precision, recall, F1, support, macro averages, and weighted averages for each aspect head.

In [4]:
aspect = pd.read_csv(OUT_DIR / 'aspect_per_class_metrics.csv')
aspect.query("aspect in ['bipolar', 'personality_disorder']")

,model,aspect,class,precision,recall,f1,support
20,ALBERT,bipolar,NONE,0.995008,0.977442,0.986147,10196
21,ALBERT,bipolar,WEAK,0.100000,0.175000,0.127273,80
22,ALBERT,bipolar,CLEAR,0.609137,0.875912,0.718563,274
23,ALBERT,bipolar,macro avg,0.568048,0.676118,0.610661,10550
24,ALBERT,bipolar,weighted avg,0.978200,0.968720,0.972684,10550
25,ALBERT,personality_disorder,NONE,0.987274,0.966329,0.976689,10276
26,ALBERT,personality_disorder,WEAK,0.118321,0.198718,0.148325,156
27,ALBERT,personality_disorder,CLEAR,0.273913,0.533898,0.362069,118
28,ALBERT,personality_disorder,macro avg,0.459836,0.566315,0.495695,10550
29,ALBERT,personality_disorder,weighted avg,0.966446,0.950142,0.957566,10550


## Entropy Triage Thresholds

This gives a concrete threshold-style review simulation. It should be framed as a routing heuristic, not a calibrated error probability.

In [5]:
triage_global = pd.read_csv(OUT_DIR / 'entropy_triage_global_thresholds.csv')
triage_status = pd.read_csv(OUT_DIR / 'entropy_triage_status_stratified.csv')
display(triage_global)
display(triage_status)

,rule,entropy_quantile,entropy_threshold,review_n,review_share,disagreement_rate_in_review,share_all_disagreements_captured
0,global_top_50pct,0.50,0.000000,53043,1.000000,0.372830,1.000000
1,global_top_30pct,0.70,0.325083,24939,0.470166,0.388227,0.489583
2,global_top_20pct,0.80,0.500402,13937,0.262749,0.516036,0.363673
3,global_top_10pct,0.90,0.673012,6513,0.122787,0.589590,0.194175
4,global_top_5pct,0.95,0.897946,2948,0.055578,0.641452,0.095621


,status,rule,entropy_threshold,review_n,review_share_within_status,disagreement_rate_in_review,share_status_disagreements_captured
0,Anxiety,within_status_top_20pct,0.587501,788,0.202675,0.265228,0.357877
1,Bipolar,within_status_top_20pct,0.639032,666,0.231491,0.716216,0.262232
2,Depression,within_status_top_20pct,0.639032,3581,0.232472,0.585311,0.234373
3,Normal,within_status_top_20pct,0.325083,6099,0.373005,0.135268,0.404214
4,Personality disorder,within_status_top_20pct,0.801819,241,0.200666,0.933610,0.199468
5,Stress,within_status_top_20pct,0.639032,589,0.220682,0.889643,0.252652
6,Suicidal,within_status_top_20pct,0.500402,2518,0.236365,0.440032,0.347662


## LLM Robustness Subset Design

This summarizes the separate GTheory mental-health subset used as evidence that robustness was examined across model families, prompt types, temperatures, and seeds.

In [6]:
robustness = pd.read_csv(OUT_DIR / 'llm_robustness_summary.csv')
design = pd.read_csv(OUT_DIR / 'llm_robustness_design_counts.csv')
pairwise = pd.read_csv(OUT_DIR / 'llm_robustness_pairwise_agreement.csv')
stability = pd.read_csv(OUT_DIR / 'llm_robustness_label_stability.csv')
display(robustness)
display(pairwise)
display(stability)
design.head()

,n_items,n_annotations,n_evaluators,n_prompt_types,n_temperatures,n_seeds,median_annotations_per_item,mean_unique_labels_per_item
0,100,21600,4,3,6,3,216.0,3.2


,condition,evaluator_a,evaluator_b,n_items,percent_agreement,cohen_kappa
0,rubric_t0_seed1001,gemini-2.5-flash,gpt-4o-mini,100,0.80,0.738460
1,rubric_t0_seed1001,gemini-2.5-flash,llama-70b,100,0.75,0.683063
2,rubric_t0_seed1001,gemini-2.5-flash,mistral-small,100,0.75,0.678828
3,rubric_t0_seed1001,gpt-4o-mini,llama-70b,100,0.78,0.715505
4,rubric_t0_seed1001,gpt-4o-mini,mistral-small,100,0.74,0.658030
5,rubric_t0_seed1001,llama-70b,mistral-small,100,0.72,0.643312


,evaluator,mean,median,min
0,gemini-2.5-flash,0.843519,0.916667,0.370370
1,gpt-4o-mini,0.910741,0.981481,0.351852
2,llama-70b,0.879259,0.962963,0.425926
3,mistral-small,0.801481,0.851852,0.333333


,evaluator,model_string,prompt_type,temperature,seed,n_annotations
0,gemini-2.5-flash,gemini-2.5-flash,cot,0.0,1001,100
1,gemini-2.5-flash,gemini-2.5-flash,cot,0.0,2001,100
2,gemini-2.5-flash,gemini-2.5-flash,cot,0.0,3001,100
3,gemini-2.5-flash,gemini-2.5-flash,cot,0.2,1001,100
4,gemini-2.5-flash,gemini-2.5-flash,cot,0.2,2001,100
